# Porn detection (Classification)

## Check
- [x] Проверить утечеку данных
- [ ] 

## Data preprocessing:
  1) Удалить строки с пустым title
  2) Разбить на леммы и морфемы
  3) Дисбаланс, будем использовать stratify 

## Secificity
1) Для url используем n-grams размером 3-7
2) Для Title используем n-grams размером 1-3

## MODELS
1) SGDclassifier (fit)
    - обычно используют на более больших выборках
    - используем: hinge, log_loss
2) SGDclassifier (partial fit)
3) LogisticRegression

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
import os
from razdel import tokenize
import pymorphy3
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, hinge_loss, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, ParameterGrid
from time import time
from scipy.sparse import hstack
version = 0

In [2]:
df = pd.read_csv("../datasets/porn_detection/train.csv")
df = df.dropna(subset=['title'])

#### Заметим, что некоторые комбинации букв содержатся только в сайтах 18+ (porn, porevo)

In [3]:
# preprocessing
def tokenize_df(df):

    def lemmatize_text(text):
        tokens = [token.text for token in tokenize(text)]
        lemmas = []
        for token in tokens:
            if token.isalpha():
                parsed = morph.parse(token)[0]  # берем первый вариант разбора
                lemmas.append(parsed.normal_form)
        return " ".join(lemmas)
    
    df['tokens'] = df['title'].apply(lambda x: " ".join([token.text for token in tokenize(x)])) 
    morph = pymorphy3.MorphAnalyzer()   # эта штука может убирать названия фильмов и тд
    df['lemmatized'] = df['tokens'].apply(lemmatize_text)
    df = df.drop(columns=["tokens"])

In [4]:
tokenize_df(df)

In [5]:
encoder = TfidfVectorizer(lowercase=True,
                          ngram_range=(1, 2),
                          min_df=2,
                          max_df=0.8, 
                          sublinear_tf=True)
encoder = TfidfVectorizer(analyzer="char_wb",
                          ngram_range=(2, 5), 
                          min_df=3, 
                          sublinear_tf=True, 
                          max_features=300_000)
encoder_url = TfidfVectorizer(analyzer="char_wb", 
                              ngram_range=(2, 5), 
                              min_df=2, 
                              sublinear_tf=True)


y = df["label"]
id_train, id_test,  = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y)

X_train = hstack([encoder.fit_transform(df["lemmatized"].iloc[id_train]), encoder_url.fit_transform(df["url"].iloc[id_train])]).tocsr()
X_test = hstack([encoder.transform(df["lemmatized"].iloc[id_test]), encoder_url.transform(df["url"].iloc[id_test])]).tocsr()
y_train = y.iloc[id_train]
y_test = y.iloc[id_test]
print(f"Размеры ТРЕНИРОВОЧНОЙ выборки {X_train.shape}")
print(f"Размеры ТЕСТОВОЙ выборки {X_test.shape}")
version+=1

Размеры ТРЕНИРОВОЧНОЙ выборки (108246, 315318)
Размеры ТЕСТОВОЙ выборки (27062, 315318)


### Fit SGDClassifier

In [6]:
version = int(input("Input the version: "))
n_samples = X_train.shape[0]
log_filename = f"log_reg_train_fit_{version}_SGD.log"

os.makedirs(f"plots_fit_{version}", exist_ok=True)

logger = logging.getLogger(f"sgd_training_fit_{version}")
logger.setLevel(logging.INFO)
if not logger.handlers:
    file_handler = logging.FileHandler(log_filename, mode="a", encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)

param_grid = {
    "loss": ["hinge", "log_loss"], 
    "penalty" : ["l1", "l2"],
    "alpha": [1e-5, 1e-6],
    "learning_rate": ["optimal"],
    "eta0": [1e-3, 1e-4], 
    "n_iter_no_change": [5, 10], 
    "tol": [1e-4, 1e-5],
    "max_iter": [1000, 2000],
    "validation_fraction": [0.1, 0.2],
    "class_weight": [dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train)))), None]
}

model_num = 0
for params in ParameterGrid(param_grid):
    time_start = time()
    model = SGDClassifier(**params,
                          random_state = 42,
                          early_stopping = True)

    print(f"Model num: {model_num}")
    logger.info(
        f"=== New model #{model_num} "
        f"params={params} ==="
    )

    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    if params["loss"] == "log_loss": 
        cur_loss_train = log_loss(y_train, model.predict_proba(X_train))
        cur_loss_test = log_loss(y_test, model.predict_proba(X_test))
    else:
        cur_loss_train = hinge_loss(y_train, model.decision_function(X_train))
        cur_loss_test = hinge_loss(y_test, model.decision_function(X_test))
    logger.info(
        f"Model {model_num}| "
        f"f1={f1_score(y_test, y_pred_test):.4f} recall={recall_score(y_test, y_pred_test):.4f} loss_train={cur_loss_train:.4f} "
        f"loss_test={cur_loss_test:.4f}"
    )

    logger.info(f"Model {model_num} | training finished | total_elapsed={(time() - time_start):.2f}s")
    model_num+=1


Model num: 0
Model num: 1
Model num: 2
Model num: 3
Model num: 4
Model num: 5
Model num: 6
Model num: 7
Model num: 8
Model num: 9
Model num: 10
Model num: 11
Model num: 12
Model num: 13
Model num: 14
Model num: 15
Model num: 16
Model num: 17
Model num: 18
Model num: 19
Model num: 20
Model num: 21
Model num: 22
Model num: 23
Model num: 24
Model num: 25
Model num: 26
Model num: 27
Model num: 28
Model num: 29
Model num: 30
Model num: 31
Model num: 32
Model num: 33
Model num: 34
Model num: 35
Model num: 36
Model num: 37
Model num: 38
Model num: 39
Model num: 40
Model num: 41
Model num: 42
Model num: 43
Model num: 44
Model num: 45
Model num: 46
Model num: 47
Model num: 48
Model num: 49
Model num: 50
Model num: 51
Model num: 52
Model num: 53
Model num: 54
Model num: 55
Model num: 56
Model num: 57
Model num: 58
Model num: 59
Model num: 60
Model num: 61
Model num: 62
Model num: 63
Model num: 64
Model num: 65
Model num: 66
Model num: 67
Model num: 68
Model num: 69
Model num: 70
Model num: 71
Mo

### Partial fit SGDClassifier

In [7]:
# version = int(input("Input the version: "))
n_samples = X_train.shape[0]

log_file_name = f"log_reg_train_part_fit_{version}_SGD.log"

os.makedirs("plots_part_fit", exist_ok=True)

logger = logging.getLogger(f"sgd_part_training_{version}")
logger.setLevel(logging.INFO)
if not logger.handlers:
    file_handler = logging.FileHandler(log_file_name, mode="a", encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)


batch_size_list = [128] 
n_iter_no_change_list = [5] 
tol_list = [1e-3, 1e-4]
max_epochs = 1000
param_grid = {
    "loss": ["hinge", "log_loss"], 
    "penalty" : ["l1", "l2"],
    "alpha": [1e-6],
    "learning_rate": ["optimal"],
    "eta0": [1e-3, 1e-4], 
    "class_weight": [None, dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train))))]
}
model_num = 0
for bs in batch_size_list:
    for n_iter_no_change in n_iter_no_change_list:
        for tol in tol_list:
             for params in ParameterGrid(param_grid):
                iters_no_change = 0
                loss_history_train, loss_history_test, f1_history, recall_history = [], [], [], []
                model = SGDClassifier(
                    **params,
                    random_state = 42,
                    warm_start = True
                    )

                print(f"Model num: {model_num}")
                logger.info(
                    f"=== New model #{model_num} | batch_size={bs}, "
                    f"n_iter_no_change={n_iter_no_change}, tol={tol}, params={params} ==="
                )
                time_start = time()

                STOP = False
                for epoch in range(max_epochs):
                    if STOP:
                        break

                    for idx in range(0, n_samples, bs):
                        start, stop = idx, min(idx+bs, n_samples)
                        X_batch = X_train[start:stop]
                        y_batch = y_train[start:stop]
                        if idx == 0 and epoch == 0:
                            model.partial_fit(X_batch, y_batch, np.unique(y_train))
                        else:
                            model.partial_fit(X_batch, y_batch)

                        if idx%(bs*1000) == 0:
                            y_pred_test = model.predict(X_test)
                            if params["loss"] == "log_loss": 
                                cur_loss_train = log_loss(y_train, model.predict_proba(X_train))
                                cur_loss_test = log_loss(y_test, model.predict_proba(X_test))
                            else:
                                cur_loss_train = hinge_loss(y_train, model.decision_function(X_train))
                                cur_loss_test = hinge_loss(y_test, model.decision_function(X_test))
                            loss_history_train.append(cur_loss_train)
                            loss_history_test.append(cur_loss_test)
                            f1_history.append(f1_score(y_test, y_pred_test))
                            recall_history.append(recall_score(y_test, y_pred_test))
                            logger.info(
                                f"Model {model_num} | epoch={epoch} idx={idx} | "
                                f"f1={f1_history[-1]:.4f} recall={recall_history[-1]:.4f} loss_train={cur_loss_train:.4f} "
                                f"loss_test={cur_loss_test:.4f}"
                            )
                            # проверка на остановку
                            if iters_no_change >= n_iter_no_change:
                                STOP = True
                                logger.info(f"Model {model_num} | early stopping at epoch={epoch} idx={idx}")
                                break
                            else:
                                if len(loss_history_train) >= 2 and tol > (loss_history_train[-2] - loss_history_train[-1]):
                                    iters_no_change += 1
                                else: 
                                    iters_no_change = 0

                    epoch_elapsed = time() - time_start
                    print(f"Эпоха №{epoch}: {epoch_elapsed}")
                    logger.info(f"Model {model_num} | epoch={epoch} finished | elapsed={epoch_elapsed:.2f}s")

                total_elapsed = time() - time_start
                print(f"Time elapsed for model {model_num}: {total_elapsed}")
                logger.info(f"Model {model_num} | training finished | total_elapsed={total_elapsed:.2f}s")

                fig, ax = plt.subplots()
                ax.plot(loss_history_train, label="train")
                ax.plot(loss_history_test, label="test")
                ax.set_xlabel("checkpoint")
                ax.set_ylabel("loss")
                ax.set_title(f"Model {model_num} loss (train vs test)")
                ax.legend()
                fig.savefig(f"plots_part_fit/model_{model_num}_loss.png")
                plt.close(fig)

                fig, ax = plt.subplots()
                ax.plot(f1_history, label="f1")
                ax.plot(recall_history, label="recall")
                ax.set_xlabel("checkpoint")
                ax.set_ylabel("score")
                ax.set_title(f"Model {model_num} f1 / recall")
                ax.legend()
                fig.savefig(f"plots_part_fit/model_{model_num}_f1.png")
                plt.close(fig)

                model_num+=1

Model num: 0
Эпоха №0: 3.124363660812378
Эпоха №1: 6.208603858947754
Эпоха №2: 9.17158317565918
Эпоха №3: 12.297327280044556
Эпоха №4: 15.217145204544067
Эпоха №5: 18.29434895515442
Эпоха №6: 21.38499140739441
Эпоха №7: 24.36593222618103
Эпоха №8: 27.41693377494812
Эпоха №9: 30.360044479370117
Эпоха №10: 30.473501443862915
Time elapsed for model 0: 30.4736270904541
Model num: 1
Эпоха №0: 1.5660698413848877
Эпоха №1: 3.0633625984191895
Эпоха №2: 4.6159327030181885
Эпоха №3: 6.196377277374268
Эпоха №4: 7.732928991317749
Эпоха №5: 9.309609413146973
Эпоха №6: 10.889292478561401
Эпоха №7: 12.403351783752441
Эпоха №8: 12.518453359603882
Time elapsed for model 1: 12.51856517791748
Model num: 2
Эпоха №0: 3.152966260910034
Эпоха №1: 6.3501176834106445
Эпоха №2: 9.537630319595337
Эпоха №3: 12.654597282409668
Эпоха №4: 15.896408081054688
Эпоха №5: 19.05633306503296
Эпоха №6: 22.17047691345215
Эпоха №7: 25.3641996383667
Эпоха №8: 25.492817640304565
Time elapsed for model 2: 25.492931127548218
Mode

### Fit LogisticRegression

In [8]:
from sklearn.linear_model import LogisticRegression


# version = int(input("Input the version: "))
n_samples = X_train.shape[0]
log_filename = f"log_reg_train_fit_loggin_{version}_LR.log"

os.makedirs(f"plots_fit_{version}", exist_ok=True)

logger = logging.getLogger(f"LR_training_fit_{version}")
logger.setLevel(logging.INFO)
if not logger.handlers:
    file_handler = logging.FileHandler(log_filename, mode="a", encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)

param_grid = {
            "l1_ratio": [0.0 ,1.0],
            "C": [1e+2, 10.0, 1.0, 2],
            "tol": [1e-5, 1e-4],
            "solver": ["lbfgs", "liblinear"],
            "max_iter": [100, 120],
            "class_weight": [dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train)))), None]
}

model_num = 0
for params in ParameterGrid(param_grid):
    time_start = time()

    print(f"Model num: {model_num}")
    logger.info(
        f"=== New model #{model_num} "
        f"params={params} ==="
    )

    try:
        model = LogisticRegression(**params,
                          random_state = 42)
        model.fit(X_train, y_train)
    except ValueError as e:
        print(f"Выбранная модель не поддерживает такой набор парметров: {params}")
        continue

    y_pred_test = model.predict(X_test)
    cur_loss_train = log_loss(y_train, model.predict_proba(X_train))
    cur_loss_test = log_loss(y_test, model.predict_proba(X_test))
    logger.info(
        f"Model {model_num}| "
        f"f1={f1_score(y_test, y_pred_test):.4f} recall={recall_score(y_test, y_pred_test):.4f} loss_train={cur_loss_train:.4f} "
        f"loss_test={cur_loss_test:.4f}"
    )

    logger.info(f"Model {model_num} | training finished | total_elapsed={(time() - time_start):.2f}s")
    model_num+=1


Model num: 0
Model num: 1
Model num: 2
Model num: 3
Model num: 4
Model num: 5
Model num: 6
Model num: 7
Model num: 8
Выбранная модель не поддерживает такой набор парметров: {'C': 100.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'l1_ratio': 1.0, 'max_iter': 100, 'solver': 'lbfgs', 'tol': 1e-05}
Model num: 8
Выбранная модель не поддерживает такой набор парметров: {'C': 100.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'l1_ratio': 1.0, 'max_iter': 100, 'solver': 'lbfgs', 'tol': 0.0001}
Model num: 8
Model num: 9
Model num: 10
Выбранная модель не поддерживает такой набор парметров: {'C': 100.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'l1_ratio': 1.0, 'max_iter': 120, 'solver': 'lbfgs', 'tol': 1e-05}
Model num: 10
Выбранная модель не поддерживает такой набор парметров: {'C': 100.0, 'class_weight': {

### Fit LinearSVC

In [9]:
from sklearn.svm import LinearSVC


# version = int(input("Input the version: "))
n_samples = X_train.shape[0]
log_filename = f"log_reg_train_fit_loggin_{version}_LSVC.log"

os.makedirs(f"plots_fit_LSVC_{version}", exist_ok=True)

logger = logging.getLogger(f"LSVC_training_fit_{version}")
logger.setLevel(logging.INFO)
if not logger.handlers:
    file_handler = logging.FileHandler(log_filename, mode="a", encoding="utf-8")
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)

param_grid = {
    "penalty": ["l1", "l2"],
    "loss": ['squared_hinge', 'hinge'], 
    "C": [3.0, 2.0, 1.0, 0.5], 
    "class_weight": [dict(zip(np.unique(y_train), y_train.shape[0] / (2 * np.bincount(y_train)))), None],
    "max_iter": [1000, 1500]
}

model_num = 0
for params in ParameterGrid(param_grid):
    time_start = time()

    print(f"Model num: {model_num}")
    logger.info(
        f"=== New model #{model_num} "
        f"params={params} ==="
    )

    try:
        model = LinearSVC(**params,
                          dual=True,
                          random_state = 42)
        model.fit(X_train, y_train)
    except ValueError as e:
        print(f"Выбранная модель не поддерживает такой набор парметров: {params}")
        continue

    y_pred_test = model.predict(X_test)
    cur_loss_train = hinge_loss(y_train, model.decision_function(X_train))
    cur_loss_test = hinge_loss(y_test, model.decision_function(X_test))
    logger.info(
        f"Model {model_num}| "
        f"f1={f1_score(y_test, y_pred_test):.4f} recall={recall_score(y_test, y_pred_test):.4f} loss_train={cur_loss_train:.4f} "
        f"loss_test={cur_loss_test:.4f}"
    )

    logger.info(f"Model {model_num} | training finished | total_elapsed={(time() - time_start):.2f}s")
    model_num+=1


Model num: 0
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'loss': 'squared_hinge', 'max_iter': 1000, 'penalty': 'l1'}
Model num: 0
Model num: 1
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'loss': 'squared_hinge', 'max_iter': 1500, 'penalty': 'l1'}
Model num: 1
Model num: 2
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'loss': 'hinge', 'max_iter': 1000, 'penalty': 'l1'}
Model num: 2


c:\Users\user_1\miniforge3\envs\ipynb_base\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model num: 3
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'loss': 'hinge', 'max_iter': 1500, 'penalty': 'l1'}
Model num: 3


c:\Users\user_1\miniforge3\envs\ipynb_base\Lib\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model num: 4
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': None, 'loss': 'squared_hinge', 'max_iter': 1000, 'penalty': 'l1'}
Model num: 4
Model num: 5
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': None, 'loss': 'squared_hinge', 'max_iter': 1500, 'penalty': 'l1'}
Model num: 5
Model num: 6
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': None, 'loss': 'hinge', 'max_iter': 1000, 'penalty': 'l1'}
Model num: 6
Model num: 7
Выбранная модель не поддерживает такой набор парметров: {'C': 3.0, 'class_weight': None, 'loss': 'hinge', 'max_iter': 1500, 'penalty': 'l1'}
Model num: 7
Model num: 8
Выбранная модель не поддерживает такой набор парметров: {'C': 2.0, 'class_weight': {np.int64(0): np.float64(0.570472416046546), np.int64(1): np.float64(4.0474872868680825)}, 'loss': 'squared_hinge', 'max_iter': 1000, 'penalty': 'l1'}
Model num: 8
Model num: 9
Выбранная модель не поддерживает такой набо